Kolmogorov-Smirnoff Test: Lilliefors Statistic

PHYS4730, Fall 2026

Lab 18 Assignment

JB's solution.

In [ ]:
%reset -f

import numpy as np
import scipy.optimize as SciOpt
import scipy.stats as SciStat
from numpy import random as rnd
from numpy import linalg as LA
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams.update({'font.size': 18})

In [ ]:
#####################
def lilliefors(Npts):
#####################

    GMEAN = 23.0                            # in principle arbitrary 
    GSIGM = 16.5                            # in principle arbitrary 
    
    rvvect = rnd.normal(GMEAN,GSIGM,Npts)   # Part 1a
    rvmean = np.mean(rvvect)                # Part 1b
    rvstdv = np.std(rvvect,ddof=1)          # Part 1b, w/ Bessel correction
    rvnorm = (rvvect-rvmean)/rvstdv         # Part 1c 
        
    epdf,x = np.histogram(rvnorm, bins=2000, range=[-10.0,10.0], density=True)
    dx     = x[1]-x[0]
    x      = np.delete(x,len(x)-1) + 0.5*dx # convert x from array of bin edges -> bin centers
    
    ecdf   = np.cumsum(epdf)*dx             # Part 1d
        
    pdf    = np.exp(-0.5*x**2.0)/np.sqrt(2.0*np.pi)
    cdf    = np.cumsum(pdf)*dx              # Part 1d

    if(False):                              # debuggin'          
        plt.figure(figsize=(10,10))

        plt.subplot(2,1,1)
        plt.plot(x,ecdf,label="empirical")
        plt.plot(x, cdf,label="theoretical")
        plt.ylabel("$CDF$")
            
        plt.subplot(2,1,2)
        plt.plot(x,np.abs(ecdf-cdf))
        plt.xlabel("$x$")
        plt.ylabel("$ECDF-CDF$")
            
        plt.show()
            
    supremum = np.amax(np.abs(ecdf-cdf))  # Part 1e
        
    return supremum                       # Part 1f

In [ ]:
###################
### main script ###
###################

### compute Lilliefors Statistic for Nsets data sets, ###
### each with Npnts RVs.                              ###

Nsets = 100000
Npnts = 12

lstat = np.zeros(Nsets)

for i in np.arange(0,Nsets):
    lstat[i] = lilliefors(Npnts)
    if(np.mod(i,10000)==0):
        print("%6d  %12.10f" % (i,lstat[i]))

In [ ]:
### write L.S. to file, read ###

import matplotlib
matplotlib.rcParams.update({'font.size': 18})

filename = "lilliefors.dat"

if(True):
    f = open(filename,"w")
    for i in range(0,len(lstat)):
        f.write("%4d  %12.10f \n" % (i,lstat[i]))
    f.close()

if(True):
    data = np.loadtxt(filename)
    lstat = data[:,1]

plt.figure(figsize=(8,8))
plt.hist(lstat,bins=40,range=[0.0,0.4])
plt.xlabel("Lilliefors Statistic")
plt.ylabel("Number of Counts")
plt.show()

In [ ]:
### compare with Table 1 of Lilliefors (1967) ###

alpha = np.array([0.20,0.15,0.10,0.05,0.01])
confl = 1.0-alpha

print("")
print("Sample Size: %3d" % Npnts)
print("")
print("Significance   Null C.L.    Cutoff:")
print("-----------------------------------")
for i in np.arange(0,len(alpha)):
    print("     %4.2f         %4.2f       %5.3f" % (alpha[i],confl[i],np.quantile(lstat,confl[i])))
print("")